# Exploring THP-1 Perturb-seq with Scanpy
This notebook adapts the [Scanpy preprocessing and clustering tutorial](https://scanpy.scverse.org/en/stable/tutorials/basics/clustering.html) to the conventional CRISPRi knockdown screen [GSM6858449](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM6858449).

The questions are simple: What is in the file? What does QC look like? How do cells cluster? Do replicate, channel, cell-cycle or perturbation labels track the embedding? Which genes characterize clusters?

These are THP-1 cells, not a PBMC mixture: **clusters are not automatically different cell types**. This is exploratory single-cell analysis, not reproduction of scMINER Fig. 6a, a new perturbation-response reference, or an SJARACNe benchmark. It does not touch the running 100-bootstrap workflow.

Default: a uniform, seeded **10,000-cell preview** from all assignments. Set `MAX_CELLS = None` for all cells, or `NT_ONLY = True` for pure non-targeting controls. The source H5AD is opened read-only, and all processing uses a new in-memory object. No files are exported unless you enable the last cell.

## 1. Environment
Select the `C:\Users\adamd\miniconda3\python.exe` kernel in VS Code/Jupyter; Scanpy 1.11.5 and its clustering dependencies were present when this notebook was created. For another environment, install `requirements_scanpy.txt` using that environment's Python, then restart the kernel. Installation is deliberately not performed by this notebook.

Two CPU threads are used to leave resources for the independent SJARACNe run. Local Numba/Matplotlib cache directories are created under `results/scanpy_runtime_cache/` in the current working directory to avoid writing into the Python installation; use `SJARACNE_THP1_SCANPY_CACHE` to change that location. These are runtime caches, not analysis exports. Package versions are shown below; a fixed seed improves repeatability but is not a guarantee of identical results across dependency versions.

In [ ]:
import os
from pathlib import Path
runtime_cache = Path(os.environ.get('SJARACNE_THP1_SCANPY_CACHE', str(Path.cwd() / 'results/scanpy_runtime_cache')))
runtime_cache.mkdir(parents=True, exist_ok=True)
for name, subdir in (('NUMBA_CACHE_DIR', 'numba'), ('MPLCONFIGDIR', 'matplotlib')):
    cache_dir = runtime_cache / subdir
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(cache_dir.resolve())
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ.setdefault(name, '2')

import sys
import json
from datetime import datetime, timezone
from importlib.metadata import version
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import numba
from threadpoolctl import threadpool_limits
from IPython.display import display

get_ipython().run_line_magic('matplotlib', 'inline')
_thread_limits = threadpool_limits(limits=2)
numba.set_num_threads(min(2, numba.config.NUMBA_NUM_THREADS))
sc.settings.n_jobs = 2
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=90, facecolor='white', frameon=False)
print('Python:', sys.executable)
display(pd.Series({name: version(name) for name in
    ('scanpy', 'anndata', 'numpy', 'pandas', 'scipy', 'igraph', 'leidenalg', 'umap-learn')}, name='version'))

## 2. Configuration
The existing `SJARACNE_THP1_DATA` variable may point to the H5AD itself, its sample directory, or the parent Perturb-seq directory. Otherwise the notebook checks the known Windows and WSL locations. It downloads nothing.

Filtering is intentionally permissive: at least 100 detected genes per cell and detection in at least 3 analyzed cells per gene. There is no mitochondrial upper cutoff by default. These are exploration settings, **not** the benchmark's scater/MAD QC.

`TARGET_SUM = 10_000` makes the new normalized matrix `ln(1 + counts per 10,000)`. The tutorial uses median-depth normalization; set `TARGET_SUM = None` to use that alternative. Neither is the benchmark's log2(CPM+1).

In [ ]:
SEED = 42
MAX_CELLS = 10_000  # None = all cells; a random preview is not a new experimental replicate.
NT_ONLY = False
MIN_GENES = 100
MIN_CELLS_PER_GENE = 3
MAX_PCT_MITO = None
TARGET_SUM = 10_000
N_HVG = 2_000
N_PCS = 50
NEIGHBOR_PCS = 30
N_NEIGHBORS = 15
LEIDEN_RESOLUTIONS = (0.2, 0.5, 1.0)
CLUSTER_KEY = 'leiden_0.5'
RUN_SCRUBLET = False  # Optional; read the doublet caveats before enabling.
FOCUS_GENES = ['MYD88', 'IRAK4', 'TOLLIP', 'NFKB1']

# Optional automation override; editing MAX_CELLS above is enough for interactive use.
limit_override = os.environ.get('SJARACNE_THP1_SCANPY_MAX_CELLS')
if limit_override is not None:
    MAX_CELLS = None if limit_override.lower() in ('all', 'none', '0') else int(limit_override)
if MAX_CELLS is not None and MAX_CELLS < 100:
    raise ValueError('Use at least 100 cells for this exploration.')

filename = 'GSM6858449_KD_conventional.h5ad'
configured = os.environ.get('SJARACNE_THP1_DATA')
if configured:
    base = Path(configured).expanduser()
    candidates = [base, base / filename, base / 'THP1_GSM6858449' / filename]
else:
    candidates = [Path('E:/SpaGRN/SJARACNe/Perturb-seq/THP1_GSM6858449') / filename,
                  Path('/mnt/e/SpaGRN/SJARACNe/Perturb-seq/THP1_GSM6858449') / filename]
H5AD = next((p.resolve() for p in candidates if p.is_file() and p.suffix.lower() == '.h5ad'), None)
if H5AD is None:
    raise FileNotFoundError('Set SJARACNE_THP1_DATA to your downloaded THP-1 H5AD or directory.')
source_stat = H5AD.stat()
SOURCE_DATA_XLSX = Path(os.environ.get('SJARACNE_THP1_FIGURE6', str(H5AD.with_name('Source_Data_Figure_6.xlsx')))).expanduser().resolve()
print('Read-only source:', H5AD)
print('Preview limit:', MAX_CELLS, '| Pure NT only:', NT_ONLY)

## 3. Inspect the full dataset before subsampling
The file's `X` is already transformed; the original integer-valued counts are in `raw.X`. **Do not run count normalization directly on the supplied `X`.** Here we read metadata first, resolve symbols from the legacy raw-variable table, then load only the selected raw-count rows.

`Biological_replicate` and `10X_channel` are real metadata fields. Channels are nested within replicate, not independent biological replicates. Original cell-cycle scores are supplied metadata; their preprocessing is not reconstructed here.

In [ ]:
backed = ad.read_h5ad(H5AD, backed='r')
try:
    if backed.raw is None:
        raise ValueError('Expected counts in raw.X; refusing to treat normalized X as counts.')
    obs_all = backed.obs.copy()
    raw_var = backed.raw.var.copy()
    symbol_field = next((k for k in ('features', '_index') if k in raw_var.columns), None)
    symbols = pd.Index(raw_var[symbol_field].astype(str) if symbol_field else raw_var.index.astype(str))
    if not symbols.is_unique or not obs_all.index.is_unique or (symbols == '').any():
        raise ValueError('Nonunique/empty symbols or cell identifiers require explicit resolution.')
    print('All cells × genes:', backed.shape, '| Raw count shape:', backed.raw.shape)
    print('Supplied X:', type(backed.X).__name__, '| Count matrix:', type(backed.raw.X).__name__)
    print('Symbol field:', symbol_field, '| First symbols:', symbols[:8].tolist())
finally:
    backed.file.close()

print('Available cell metadata:', obs_all.columns.tolist())
metadata_cols = [k for k in ('Biological_replicate', '10X_channel', 'Guides',
    'Guides_collapsed_by_gene', 'Total_number_of_guides', 'Total_RNA_count',
    'Total_unique_genes', 'Percent_mitochondrial_reads', 'Cell_cycle_phase') if k in obs_all]
display(obs_all[metadata_cols].head(6))

### What perturbation labels mean
A literal `--` separates collapsed assignments. A single hyphen inside a symbol such as `ARPC4-TTLL3` is **not** a separator. Repeated assignments to the same target still define one distinct target; this does not mean the cell received only one guide.

`NT` below means every assigned collapsed target is `non-targeting`. The separate `safe-targeting` control is labeled `safe_control`, not a biological target. Control-plus-target, mixed-control and distinct-target mixtures remain separate; we do not silently call them clean single-gene perturbations. These full-metadata counts are not affected by the preview.

In [ ]:
def parse_targets(value):
    if pd.isna(value):
        return ()
    return tuple(dict.fromkeys(t.strip() for t in str(value).split('--') if t.strip()))

def assignment_class(targets):
    if not targets:
        return 'unassigned'
    if targets == ('non-targeting',):
        return 'NT'
    if targets == ('safe-targeting',):
        return 'safe_control'
    controls = {'non-targeting', 'safe-targeting'}
    if set(targets) <= controls:
        return 'mixed_controls'
    if set(targets) & controls:
        return 'control_plus_target'
    return 'single_target' if len(targets) == 1 else 'multiple_targets'

if 'Guides_collapsed_by_gene' not in obs_all:
    raise KeyError('This sample needs Guides_collapsed_by_gene metadata.')
target_tokens = obs_all['Guides_collapsed_by_gene'].map(parse_targets)
obs_all['assignment_class'] = pd.Categorical(target_tokens.map(assignment_class))
obs_all['single_target'] = target_tokens.map(lambda ts: ts[0] if len(ts) == 1 else 'multiple/unassigned')
display(obs_all['assignment_class'].value_counts().rename('all_cells'))
display(obs_all.loc[obs_all['assignment_class'] == 'single_target', 'single_target']
        .value_counts().head(20).rename('top_target_cell_counts'))
if 'Biological_replicate' in obs_all:
    display(pd.crosstab(obs_all['Biological_replicate'], obs_all['assignment_class']))
if '10X_channel' in obs_all and 'Biological_replicate' in obs_all:
    display(pd.crosstab(obs_all['10X_channel'], obs_all['Biological_replicate']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
obs_all['assignment_class'].value_counts().plot.bar(ax=axes[0], rot=40, color='steelblue')
axes[0].set(title='All cells: assignment classes', ylabel='Cells', xlabel='')
if 'Total_number_of_guides' in obs_all:
    obs_all['Total_number_of_guides'].value_counts().sort_index().plot.bar(ax=axes[1], color='darkorange')
    axes[1].set(title='All cells: detected guide counts', xlabel='Guides per cell', ylabel='Cells')
fig.tight_layout()
plt.show()

## 4. Create a sparse, count-based analysis object
Uniform sampling is without replacement and independent of labels. Rare perturbations can have few or no cells in the preview. Gene detection filters and learned embeddings refer to the selected cells, not necessarily the full screen.

Only this fresh object is processed. We keep the raw symbols as `var_names`, avoiding the source file's numeric legacy variable names. Stored count/expression matrices remain sparse. The marker-testing implementation uses bounded dense gene chunks internally; our expression plots extract only individual columns, not a dense copy of the whole matrix.

In [ ]:
def choose_rows(obs, max_cells, nt_only, seed):
    candidates = np.flatnonzero((obs['assignment_class'] == 'NT').to_numpy()) if nt_only else np.arange(len(obs))
    if max_cells is not None and len(candidates) > max_cells:
        candidates = np.random.default_rng(seed).choice(candidates, max_cells, replace=False)
    return np.sort(candidates)

rows = choose_rows(obs_all, MAX_CELLS, NT_ONLY, SEED)
if len(rows) < 100:
    raise ValueError('Too few selected cells for this workflow.')
backed = ad.read_h5ad(H5AD, backed='r')
try:
    counts = sparse.csr_matrix(backed.raw.X[rows, :])
finally:
    backed.file.close()
counts.eliminate_zeros()
if not np.isfinite(counts.data).all() or (counts.data < 0).any() or not np.array_equal(counts.data, np.rint(counts.data)):
    raise ValueError('raw.X must contain finite, nonnegative, integer-valued counts.')
if counts.data.size and counts.data.max() >= 2**24:
    raise ValueError('Counts exceed the exact-integer float32 range; retain float64 instead.')
counts = counts.astype(np.float32)
# '_index' is a legacy symbol column reserved by modern AnnData writers.
var = raw_var.rename(columns={'_index': 'source_gene_symbol'}).copy()
var['original_var_index'] = var.index.astype(str)
var.index = pd.Index(symbols, name='gene_symbol')
adata = ad.AnnData(X=counts, obs=obs_all.iloc[rows].copy(), var=var)
del counts
for key in ('Biological_replicate', '10X_channel', 'Cell_cycle_phase', 'single_target'):
    if key in adata.obs:
        adata.obs[key] = pd.Categorical(adata.obs[key])
print(adata)
print('Count CSR storage (MiB):', round((adata.X.data.nbytes + adata.X.indices.nbytes + adata.X.indptr.nbytes) / 2**20, 1))
display(pd.concat([obs_all['assignment_class'].value_counts().rename('all_cells'),
                   adata.obs['assignment_class'].value_counts().rename('selected_cells')], axis=1).fillna(0))

## 5. Count-based QC
Inspect depth, detected genes and mitochondrial percentage together, including differences across replicates. High mitochondrial fractions or extreme depths are flags for investigation, not proof of poor quality. Perturbations can alter these distributions too. The prefix-based ribosomal flag is a convenient heuristic; the hemoglobin flag uses an explicit symbol list.

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
adata.var['ribo'] = adata.var_names.str.startswith(('RPS', 'RPL'))
adata.var['hb'] = adata.var_names.isin(['HBA1', 'HBA2', 'HBB', 'HBD', 'HBE1', 'HBG1', 'HBG2', 'HBM', 'HBQ1', 'HBZ'])
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo', 'hb'], percent_top=None, log1p=True, inplace=True)
qc_fields = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
display(adata.obs[qc_fields].describe(percentiles=[0.01, 0.5, 0.99]).T)
if 'Total_RNA_count' in adata.obs:
    delta = np.abs(adata.obs['total_counts'].to_numpy() - pd.to_numeric(adata.obs['Total_RNA_count']).to_numpy())
    print('Maximum raw-total difference from supplied metadata:', delta.max())
group = 'Biological_replicate' if 'Biological_replicate' in adata.obs else None
sc.pl.violin(adata, qc_fields, groupby=group, multi_panel=True, jitter=False)
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color='pct_counts_mt')

In [ ]:
before_filter = adata.shape
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
if MAX_PCT_MITO is not None:
    adata = adata[adata.obs['pct_counts_mt'] <= MAX_PCT_MITO].copy()
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
if adata.n_obs < 100 or adata.n_vars < 100:
    raise ValueError('Too little data remains; reconsider the exploratory filtering settings.')
print('Before:', before_filter, '| After:', adata.shape)
print('Cells removed:', before_filter[0] - adata.n_obs, '| Genes removed:', before_filter[1] - adata.n_vars)
print('Mitochondrial upper cutoff:', MAX_PCT_MITO)
# QC metrics describe libraries before the gene filter; do not reinterpret them as post-filter totals.

### Optional: Scrublet doublet scores
The tutorial includes doublet detection. It is off here by default to keep the first exploration light and avoid automatically removing perturbed states. If enabled, scores are computed **before normalization**, separately by `10X_channel` (capture channel, not biological replicate), and cells are not automatically removed.

On a random preview, fewer cells and altered observed doublet proportions can change calls. Strong perturbation-induced states can complicate interpretation. Neither multiple guide assignments nor a high score alone proves a physical doublet. Use the full dataset and investigate channel-specific expectations before acting on calls. The helper refuses channels with fewer than 100 analyzed cells. See the [Scrublet API](https://scanpy.scverse.org/en/stable/api/generated/scanpy.pp.scrublet.html).

In [ ]:
if RUN_SCRUBLET:
    capture_key = '10X_channel' if '10X_channel' in adata.obs else None
    if capture_key and (adata.obs[capture_key].value_counts() < 100).any():
        raise ValueError('A capture channel has <100 cells; use the full dataset for this optional step.')
    sc.pp.scrublet(adata, batch_key=capture_key, random_state=SEED)
    display(adata.obs[['doublet_score', 'predicted_doublet']].head())
    sc.pl.scrublet_score_distribution(adata)
else:
    print('Scrublet skipped; no doublet filtering was performed.')

## 6. Normalize and select variable genes
A count layer is saved before normalization. The new `X` contains log-normalized values on the retained genes; it is not guaranteed to match the supplied transformed `X`. We keep all retained genes for marker inspection, but use 2,000 highly variable genes (HVGs) for PCA.

HVG selection accounts for the two biological replicates when both are present. This is **feature selection, not batch correction**. We do not regress out perturbations, cell-cycle scores or batch labels; doing so could remove the signal we want to inspect.

In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
sc.pp.log1p(adata)
hvg_batch = 'Biological_replicate' if ('Biological_replicate' in adata.obs and adata.obs['Biological_replicate'].nunique() > 1) else None
sc.pp.highly_variable_genes(adata, n_top_genes=min(N_HVG, adata.n_vars), flavor='seurat', batch_key=hvg_batch)
print('HVGs:', int(adata.var['highly_variable'].sum()), '| Batch-aware HVG key:', hvg_batch)
sc.pl.highly_variable_genes(adata)
hvg_fields = [k for k in ('means', 'dispersions_norm', 'highly_variable_nbatches') if k in adata.var]
display(adata.var.loc[adata.var['highly_variable'], hvg_fields].sort_values('dispersions_norm', ascending=False).head(12))

## 7. PCA
PCA uses only HVGs while leaving the all-gene matrix available. We follow the tutorial's direct PCA on log-normalized values, without dense all-gene scaling. Compare the first PCs with replicate and QC: visible separation is a diagnostic clue, not a reason to immediately apply integration.

In [ ]:
pca_components = min(N_PCS, int(adata.var['highly_variable'].sum()) - 1, adata.n_obs - 1)
if pca_components < 3:
    raise ValueError('Need more variable genes/cells for PCA.')
sc.pp.pca(adata, n_comps=pca_components, mask_var='highly_variable', svd_solver='arpack', random_state=SEED)
sc.pl.pca_variance_ratio(adata, n_pcs=pca_components, log=True)
pca_colors = [k for k in ('Biological_replicate', 'pct_counts_mt', 'log1p_total_counts', 'Cell_cycle_phase') if k in adata.obs]
sc.pl.pca(adata, color=pca_colors, ncols=2, size=4)

## 8. Neighbors, UMAP and Leiden clustering
The neighbor graph uses the first 30 PCs (or all available if fewer). Leiden partitions that graph; UMAP is only a visualization of it. Three resolutions illustrate how cluster granularity depends on the chosen parameter. UMAP islands, distances and cluster counts are not direct evidence of different cell identities or causal perturbation effects.

In [ ]:
neighbor_pcs = min(NEIGHBOR_PCS, pca_components)
sc.pp.neighbors(adata, n_neighbors=min(N_NEIGHBORS, adata.n_obs - 1), n_pcs=neighbor_pcs, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
cluster_keys = []
for resolution in LEIDEN_RESOLUTIONS:
    key = f'leiden_{resolution:g}'
    sc.tl.leiden(adata, resolution=resolution, key_added=key, flavor='igraph',
                 directed=False, n_iterations=2, random_state=SEED)
    cluster_keys.append(key)
display(pd.Series({key: adata.obs[key].nunique() for key in cluster_keys}, name='clusters'))
sc.pl.umap(adata, color=cluster_keys, legend_loc='on data', ncols=3, size=4)

In [ ]:
overlay_keys = [k for k in ('Biological_replicate', '10X_channel', 'assignment_class',
    'Cell_cycle_phase', 'log1p_total_counts', 'pct_counts_mt') if k in adata.obs]
sc.pl.umap(adata, color=overlay_keys, ncols=2, wspace=0.5, size=4)
if RUN_SCRUBLET:
    sc.pl.umap(adata, color=['doublet_score', 'predicted_doublet'], size=4)
display(pd.crosstab(adata.obs[CLUSTER_KEY], adata.obs['Biological_replicate'], normalize='index').round(3))

### Figure 6g,h labels: a different dataset
`Source_Data_Figure_6.xlsx`, sheet `Figure_6g,h`, contains `Cell_Barcode`, `Cell_Type` and the authors' `UMAP_1`/`UMAP_2` coordinates. However, these are **PBMC CITE-seq cells (GSE213282), not THP-1 Perturb-seq cells (GSM6858449)**. The [paper's Fig. 6 caption and CITE-seq methods](https://www.nature.com/articles/s41467-025-59620-6) distinguish the Perturb-seq panels a-e from the CITE-seq panels f-h; the latter dataset has 4,602 cells of five cell types.

The cells below import the labels, count exact cell-ID matches against both the full THP-1 file and the analyzed subset, and show the labels on the **published PBMC UMAP only**. No labels are assigned to `adata.obs`, and `adata.obsm['X_umap']` is not replaced. Barcode strings can recur across unrelated 10x experiments; stripping channel/library suffixes or matching by row order cannot make this a valid annotation source. In the original files there are zero exact matches.

The workbook is read-only and defaults to the H5AD's directory. Set `SJARACNE_THP1_FIGURE6` if it is elsewhere. If the default workbook is absent, this optional section is skipped; an explicitly configured missing path is an error. The blank barcode/label row is the spreadsheet's second header (Protein/Activity/Expression), not a cell.

In [ ]:
def prepare_figure6gh_labels(frame):
    columns = ['Cell_Barcode', 'Cell_Type', 'UMAP_1', 'UMAP_2']
    missing = set(columns) - set(frame.columns)
    if missing:
        raise ValueError(f'Figure_6g,h is missing columns: {sorted(missing)}')
    # Drop the second header, never a partially specified cell record.
    table = frame.loc[frame[columns].notna().any(axis=1), columns].copy()
    if table.empty or table[['Cell_Barcode', 'Cell_Type']].isna().any().any():
        raise ValueError('Expected nonempty, fully labeled cell records.')
    for column in ('Cell_Barcode', 'Cell_Type'):
        table[column] = table[column].astype(str).str.strip()
        if (table[column] == '').any():
            raise ValueError(f'Empty {column} in Figure_6g,h.')
    if table['Cell_Barcode'].duplicated().any():
        raise ValueError('Duplicate published cell identifiers; refusing an ambiguous mapping.')
    for column in ('UMAP_1', 'UMAP_2'):
        table[column] = pd.to_numeric(table[column], errors='raise')
    if not np.isfinite(table[['UMAP_1', 'UMAP_2']].to_numpy(dtype=float)).all():
        raise ValueError('Published UMAP coordinates must be finite.')
    return table.set_index('Cell_Barcode', verify_integrity=True)

published_labels = pd.DataFrame()
excel_source_stat = None
figure6_annotation_audit = {'workbook': str(SOURCE_DATA_XLSX), 'sheet': 'Figure_6g,h',
    'source_dataset': 'GSE213282: PBMC CITE-seq',
    'target_dataset': 'GSM6858449: THP-1 conventional CRISPRi',
    'same_dataset': False, 'annotation_applied': False, 'status': 'workbook_not_loaded'}
if SOURCE_DATA_XLSX.is_file():
    excel_source_stat = SOURCE_DATA_XLSX.stat()
    published_labels = prepare_figure6gh_labels(
        pd.read_excel(SOURCE_DATA_XLSX, sheet_name='Figure_6g,h', engine='openpyxl'))
    figure6_annotation_audit.update({
        'published_cells': len(published_labels), 'published_cell_types': published_labels['Cell_Type'].nunique(),
        'thp1_full_cells': len(obs_all), 'thp1_analyzed_cells': adata.n_obs,
        'exact_matches_full': int(obs_all.index.isin(published_labels.index).sum()),
        'exact_matches_analyzed': int(adata.obs_names.isin(published_labels.index).sum()),
        'status': 'incompatible_datasets_no_label_transfer'})
    display(pd.Series(figure6_annotation_audit, name='annotation_source_audit'))
    display(published_labels['Cell_Type'].value_counts().rename('published_PBMC_cells'))
    print('These labels belong to a different experiment. No cell-type labels were added to THP-1.')
elif os.environ.get('SJARACNE_THP1_FIGURE6'):
    raise FileNotFoundError(f'Configured source-data workbook is missing: {SOURCE_DATA_XLSX}')
else:
    print('Optional source-data workbook not found; published-label section skipped:', SOURCE_DATA_XLSX)

In [ ]:
if not published_labels.empty:
    fig, ax = plt.subplots(figsize=(7.5, 5))
    source_types = sorted(published_labels['Cell_Type'].unique())
    palette = dict(zip(source_types, sns.color_palette('colorblind', n_colors=len(source_types))))
    for cell_type, group in published_labels.groupby('Cell_Type', sort=True):
        ax.scatter(group['UMAP_1'], group['UMAP_2'], s=6, alpha=0.7, linewidths=0,
                   color=palette[cell_type], label=f'{cell_type} (n={len(group):,})', rasterized=True)
    ax.set(title='Published PBMC CITE-seq UMAP (GSE213282)\nFigure 6g,h source data - NOT the THP-1 embedding',
           xlabel='Published UMAP 1', ylabel='Published UMAP 2')
    ax.legend(title='Published cell type', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
    ax.set_aspect('equal', adjustable='datalim')
    fig.tight_layout()
    plt.show()

## 9. Overlay a few perturbations
Plotting all target labels would create an unreadable legend. Instead we highlight a few chosen **single-distinct-target** assignments, pure NT controls and the remaining categories. Exact token parsing avoids including mixed MYD88 assignments by accident. Inspect the cell counts next to the plot; a sparse target group cannot support strong conclusions.

The full-screen target-count table above is better for coverage questions than this random preview. Enrichment of a guide group in a cluster can reflect perturbation, channel, quality, proliferation or sampling—not just the intended molecular effect.

In [ ]:
view = adata.obs['assignment_class'].astype(str).replace({'single_target': 'other single-target'})
for gene in FOCUS_GENES:
    clean_gene = (adata.obs['assignment_class'] == 'single_target') & (adata.obs['single_target'] == gene)
    view.loc[clean_gene] = gene
adata.obs['perturbation_view'] = pd.Categorical(view)
display(adata.obs['perturbation_view'].value_counts().rename('analyzed_cells'))
sc.pl.umap(adata, color='perturbation_view', size=5)
composition = pd.crosstab(adata.obs[CLUSTER_KEY], adata.obs['assignment_class'], normalize='index')
composition.plot.bar(stacked=True, figsize=(9, 3.5), rot=0)
plt.ylabel('Fraction within cluster')
plt.title('Assignment composition, not perturbation enrichment significance')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 10. Inspect genes and cluster markers
We do not transfer the tutorial's PBMC cell-type labels to this cell line. The small gene panels below are illustrative starting points for inspecting expression, not validated THP-1 state signatures. Missing genes are listed rather than silently invented. Dot size shows expression prevalence and dot color shows average log-normalized expression.

Cluster-versus-rest Wilcoxon tests are descriptive marker discovery on the same cells used to construct clusters. Their adjusted p-values are **not** a replicate-aware perturbation test, proof of causal response, or a regulatory-edge ground truth. See [Scanpy marker-test documentation](https://scanpy.scverse.org/en/stable/generated/scanpy.tl.rank_genes_groups.html).

In [ ]:
gene_panels = {'Monocytic-associated examples': ['LYZ', 'CD14', 'S100A8', 'S100A9'],
               'Response-related examples': ['NFKBIA', 'TNFAIP3', 'IL1B'],
               'Proliferation examples': ['MKI67', 'TOP2A', 'PCNA']}
present_panels = {label: [g for g in genes if g in adata.var_names] for label, genes in gene_panels.items()}
present_panels = {label: genes for label, genes in present_panels.items() if genes}
print('Panel genes absent after filtering:', [g for genes in gene_panels.values() for g in genes if g not in adata.var_names])
if present_panels:
    sc.pl.dotplot(adata, present_panels, groupby=CLUSTER_KEY, use_raw=False)
expression_genes = [g for g in ['MYD88', 'NFKBIA', 'MKI67'] if g in adata.var_names]
if expression_genes:
    sc.pl.umap(adata, color=expression_genes, use_raw=False, ncols=3, size=4)

In [ ]:
cluster_sizes = adata.obs[CLUSTER_KEY].value_counts()
marker_groups = [group for group in adata.obs[CLUSTER_KEY].cat.categories if cluster_sizes[group] >= 3]
if len(marker_groups) >= 2:
    sc.tl.rank_genes_groups(adata, groupby=CLUSTER_KEY, groups=marker_groups, reference='rest',
                           method='wilcoxon', tie_correct=True, use_raw=False, n_genes=50, key_added='cluster_markers')
    marker_table = sc.get.rank_genes_groups_df(adata, group=None, key='cluster_markers')
    display(marker_table.groupby('group', observed=True).head(5))
    sc.pl.rank_genes_groups_dotplot(adata, key='cluster_markers', n_genes=3,
                                  groupby=CLUSTER_KEY, dendrogram=False, use_raw=False)
else:
    marker_table = pd.DataFrame()
    print('Not enough nontrivial clusters for cluster-versus-rest marker testing.')

## 11. A small MYD88 sanity check
Compare MYD88 expression in pure NT cells and single-distinct-target MYD88 cells **within the analyzed preview**. Only one sparse gene column is converted to a dense vector. These groups are not matched by guide, channel or biological replicate; small preview groups and different channel mixtures can distort the comparison.

Lower target RNA would be consistent with CRISPRi knockdown, but this descriptive plot cannot establish the proportion of successfully perturbed cells. Zero expression in a cell is not proof of successful knockdown, and RNA measurements are not protein or functional measurements. This is not the scMINER response-label calculation.

In [ ]:
focus_gene = 'MYD88'
if focus_gene in adata.var_names:
    compare_mask = (adata.obs['assignment_class'] == 'NT') | (
        (adata.obs['assignment_class'] == 'single_target') & (adata.obs['single_target'] == focus_gene))
    compare = adata.obs.loc[compare_mask, ['single_target', 'Biological_replicate']].copy()
    compare['group'] = np.where(compare['single_target'].astype(str) == 'non-targeting', 'NT', focus_gene)
    gene_position = adata.var_names.get_loc(focus_gene)
    compare['log_normalized_expression'] = adata.X[compare_mask.to_numpy(), gene_position].toarray().ravel()
    compare['detected'] = compare['log_normalized_expression'] > 0
    display(compare.groupby(['Biological_replicate', 'group'], observed=True).agg(
        cells=('detected', 'size'), detected_fraction=('detected', 'mean'),
        mean_log_expression=('log_normalized_expression', 'mean')))
    if compare['group'].nunique() == 2:
        plt.figure(figsize=(6, 3.5))
        sns.violinplot(data=compare, x='group', y='log_normalized_expression', hue='Biological_replicate',
                       inner='quart', cut=0, density_norm='width')
        plt.title('MYD88: descriptive, unmatched preview comparison')
        plt.ylabel('ln(1 + normalized counts)')
        plt.tight_layout()
        plt.show()
    else:
        print('This preview does not contain both groups. Use more cells for the comparison.')
else:
    print('MYD88 was not retained by the gene filter.')

## 12. Record settings; optionally save a new result
The source file's size and modification time are checked again. A result export is **off by default**. If enabled from inside the SJARACNe checkout, it creates a fresh timestamped directory under the ignored `benchmarks/thp1_perturbseq/results/scanpy_exploration/`, never the benchmark's result directory or original H5AD. Saving the normalized matrix plus counts can produce a large file.

To explore only controls, change `NT_ONLY = True` and rerun from the start in a fresh kernel. The benchmark goes further by building NT metacells and inference networks; none of those steps is performed here.

In [ ]:
settings = {'source': str(H5AD), 'source_cells': len(obs_all), 'selected_cells': len(rows),
    'analyzed_cells': adata.n_obs, 'analyzed_genes': adata.n_vars, 'seed': SEED, 'max_cells': MAX_CELLS,
    'nt_only': NT_ONLY, 'min_genes': MIN_GENES, 'min_cells_per_gene': MIN_CELLS_PER_GENE,
    'max_pct_mito': MAX_PCT_MITO, 'target_sum': TARGET_SUM, 'hvg_count': int(adata.var['highly_variable'].sum()),
    'hvg_batch_key': hvg_batch, 'pca_components': pca_components, 'neighbor_pcs': neighbor_pcs,
    'leiden_resolutions': list(LEIDEN_RESOLUTIONS), 'scrublet_run': RUN_SCRUBLET,
    'scanpy_version': version('scanpy'), 'python': sys.executable,
    'figure6gh_annotation_audit': figure6_annotation_audit,
    'scope': 'exploratory Scanpy analysis, not benchmark inputs or perturbation ground truth'}
display(pd.Series(settings, name='exploration_settings'))
current_stat = H5AD.stat()
assert (current_stat.st_size, current_stat.st_mtime_ns) == (source_stat.st_size, source_stat.st_mtime_ns), 'Source file changed during exploration.'
if excel_source_stat is not None:
    current_excel_stat = SOURCE_DATA_XLSX.stat()
    assert (current_excel_stat.st_size, current_excel_stat.st_mtime_ns) == (excel_source_stat.st_size, excel_source_stat.st_mtime_ns), 'Source workbook changed during exploration.'

SAVE_RESULTS = False
if SAVE_RESULTS:
    repo = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'benchmarks/thp1_perturbseq').is_dir()), None)
    if repo is None:
        raise RuntimeError('Run from inside the SJARACNe checkout to use this optional export.')
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    destination = repo / 'benchmarks/thp1_perturbseq/results/scanpy_exploration' / stamp
    destination.mkdir(parents=True, exist_ok=False)
    adata.write_h5ad(destination / 'thp1_scanpy_exploration.h5ad', compression='gzip')
    if not marker_table.empty:
        marker_table.to_csv(destination / 'cluster_markers.tsv', sep='	', index=False)
    with (destination / 'settings.json').open('x', encoding='utf-8') as handle:
        json.dump(settings, handle, indent=2)
    print('New results:', destination)
else:
    print('No result files exported. The original H5AD remains unchanged.')

## What to look at next

- Does replicate/channel separation remain prominent after accounting for QC and cell-cycle overlays? Do not call it purely technical without examining perturbation composition.
- Do cluster markers suggest expression states worth inspecting, rather than assigning PBMC identities to a cell line?
- Do conclusions survive more cells, a different seed, and nearby clustering resolutions? Preview islands can be sampling artifacts.
- If a specific perturbation is interesting, rerun with enough cells and compare within channels/replicates before doing formal response testing.

The [original data/figure walkthrough](01_data_and_figure6_walkthrough.ipynb) covers the workbook and FR-Perturb files. The [author-style reconstruction](AUTHOR_RECONSTRUCTION.md) documents the separate scMINER benchmark. This notebook changes neither.